# 08 — Cross-asset regime synthesis

The most ambitious thread: do the **commodity volatility regimes** (finding 7)
line up with **stock–bond correlation regimes** (from the prior
[stock-bond study](https://github.com/carlinpercha-cpu/stock-bond-correlation))?
If a common "macro regime" drives both, that's a genuine synthesis linking the
two projects.

**Self-contained by design:** rather than depend on the other repo's files, this
notebook regenerates the stock–bond correlation signal independently from FRED
(equity index + 10Y Treasury total-return proxies), then aligns it to this
project's commodity vol regime. Everything reproducible from one FRED key.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np, pandas as pd
import requests
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
import config
plt.rcParams["figure.figsize"]=(12,5)

FRED_KEY = os.environ.get("FRED_API_KEY")
assert FRED_KEY, "FRED_API_KEY needed to regenerate the stock-bond signal"

def fred(series, start="1985-01-01"):
    r = requests.get("https://api.stlouisfed.org/fred/series/observations",
        params={"series_id":series,"api_key":FRED_KEY,"file_type":"json",
                "observation_start":start}, timeout=30)
    r.raise_for_status()
    o = pd.DataFrame(r.json()["observations"])
    o["date"]=pd.to_datetime(o["date"]); o["value"]=pd.to_numeric(o["value"],errors="coerce")
    return o.set_index("date")["value"].rename(series)

## Regenerate the stock–bond correlation signal

Use S&P 500 (SP500) and the 10Y Treasury yield (DGS10) from FRED. Bond *price*
returns move inverse to yield changes, so we use −Δyield as the bond-return
proxy. Then a rolling correlation of equity returns vs bond-return proxy — the
classic stock–bond correlation whose regime flips underpin 60/40.

In [ ]:
# SP500 on FRED only goes back ~10y; use Wilshire 5000 (WILL5000IND) for longer history
try:
    eq = fred("WILL5000IND")
except Exception:
    eq = fred("SP500")
y10 = fred("DGS10")

df = pd.concat([eq.rename("eq"), y10.rename("y10")], axis=1).dropna()
eq_ret = np.log(df["eq"].where(df["eq"]>0)).diff()
bond_ret = -df["y10"].diff()          # bond price up when yield falls
sb = pd.concat([eq_ret.rename("eq"), bond_ret.rename("bond")], axis=1).dropna()
sb_corr = sb["eq"].rolling(126).corr(sb["bond"])  # 6mo rolling stock-bond corr
print("stock-bond corr series:", sb_corr.dropna().shape[0], "days,",
      sb_corr.dropna().index.min().date(), "->", sb_corr.dropna().index.max().date())

fig,ax=plt.subplots()
sb_corr.plot(ax=ax,color="darkgreen",lw=0.8)
ax.axhline(0,color="k",lw=0.6)
ax.set_title("Regenerated stock-bond rolling 6mo correlation")
ax.set_ylabel("corr"); plt.tight_layout(); plt.show()

## Build the commodity volatility regime (this project's signal)

In [ ]:
monthly = pd.read_parquet(config.DATA/"commodities_monthly.parquet")
CORE=["gold","silver","wti","brent","natgas","copper","palladium","platinum"]
core_ret = np.log(monthly[CORE].where(monthly[CORE]>0)).diff()
mkt_vol = core_ret.rolling(12).std().mean(axis=1)
cutoff = mkt_vol.loc[:config.EXPLORE_END].quantile(0.70)
commodity_hivol = (mkt_vol>cutoff)
print("commodity high-vol months:", int(commodity_hivol.sum()))

## Align the two signals at monthly frequency

Resample the stock-bond correlation to month-end, join to the commodity vol
regime, and ask: when commodities are turbulent, is the stock-bond correlation
systematically different (e.g. more positive — the 'everything sells off
together' regime)?

In [ ]:
sb_monthly = sb_corr.resample("ME").last().rename("stock_bond_corr")
aligned = pd.concat([sb_monthly, mkt_vol.rename("commodity_vol"),
                     commodity_hivol.rename("commodity_hivol")], axis=1).dropna()

# stock-bond correlation conditional on commodity regime
by_regime = aligned.groupby("commodity_hivol")["stock_bond_corr"].agg(["mean","std","count"])
by_regime.index = ["commodity_calm","commodity_hivol"]
print("Stock-bond correlation, split by commodity volatility regime:")
print(by_regime.round(3).to_string())

# correlation between the two continuous signals
rho = aligned["stock_bond_corr"].corr(aligned["commodity_vol"])
print(f"\ncorr(stock-bond correlation, commodity vol level) = {rho:+.3f}")

In [ ]:
# visual: stock-bond correlation with commodity high-vol periods shaded
fig,ax=plt.subplots()
aligned["stock_bond_corr"].plot(ax=ax,color="darkgreen",label="stock-bond corr")
ax.axhline(0,color="k",lw=0.6)
ax.fill_between(aligned.index, aligned["stock_bond_corr"].min(), aligned["stock_bond_corr"].max(),
                where=aligned["commodity_hivol"], alpha=0.15, color="red",
                label="commodity high-vol")
ax.set_title("Stock-bond correlation vs commodity volatility regime")
ax.legend(); plt.tight_layout(); plt.show()

## Verdict (fill after running)

- **Conditional means:** is the stock-bond correlation systematically higher (more
  positive) when commodities are turbulent?
- **Continuous corr:** does commodity vol co-move with the stock-bond correlation?
- **Visual:** do the shaded commodity-turbulence periods line up with stock-bond
  correlation regime shifts (esp. the post-2021 flip to positive)?

**If aligned:** a single latent 'macro stress regime' drives both — commodities
get volatile AND the stock-bond hedge weakens together. That's the synthesis, and
a genuinely novel link between the two projects. **If not:** the regimes are
asset-class-specific, which is itself worth documenting (commodity turbulence !=
cross-asset stress). Either outcome is a real finding.